# 03 — Patient vectorization (issue #4)

Turns each patient's medical-code history into a numeric vector -- `ehrgraph.graphs.vectorize.build_patient_vectors`. This is the step right before graph construction (issue #5): a patient's row here becomes one side of the bipartite patient-code graph that gets projected down into a patient-patient similarity graph.

**Two independent settings, not one locked choice** (PROJECT_GUIDE.md item 6):
- `dedup` -- what counts as "one occurrence": every event row (`per_occurrence`), one per visit (`per_visit`), or just presence/absence (`per_patient`).
- `weighting` -- how a (deduped) count becomes a vector value: `binary` (0/1), `count` (raw count), `count_normalized` (relative frequency within that patient).

**Primary reported configuration: `binary` + `per_patient`.** Presence/absence only is *structurally immune* to the "isn't this just measuring how often the patient was seen" confound -- a patient seen 50 times and a patient seen twice get the same 1 for a diagnosis they both have, full stop. The other 8 combinations exist as a genuine sweep axis for later work (the homophily-vs-gain regression), not as unused options.

Uses the real `guo_los` cohort against the full official dataset.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

In [ ]:
import numpy as np
import pandas as pd

from ehrgraph.data.loader import open_database
from ehrgraph.graphs.vectorize import DEDUPS, WEIGHTINGS, build_patient_vectors

## Cohort: `guo_los`, filtered to patients present in the meds_reader database

In [ ]:
labels = pd.read_csv(assets / "benchmark" / "guo_los" / "labeled_patients.csv")
db = open_database(str(meds_reader_path))
db_ids = set(db)
patient_ids = [p for p in labels["patient_id"].unique().tolist() if p in db_ids]
print(f"{len(patient_ids)} patients (of {labels['patient_id'].nunique()} in the task's labels)")

## Primary configuration, per relation type

PROJECT_GUIDE.md's pipeline vectorizes diagnosis/drug/procedure separately ("per relation type") -- they stay as three separate matrices through all of Project 1's network analysis, and only collapse into one combined graph for Project 2's modeling (item 8).

In [ ]:
# Note: meds_reader's Event.table uses its own short names, not the full
# OMOP CDM table names -- verified against real data just now: procedures
# come through as "procedure", not "procedure_occurrence" (the latter
# raises no error, it just silently matches zero events, which is exactly
# the kind of silent-empty-result bug worth catching by actually running
# this rather than assuming the OMOP table name carries over unchanged).
RELATION_TABLES = ["condition", "drug_exposure", "procedure"]

primary_matrices = {}
for table in RELATION_TABLES:
    matrix, pids, codes = build_patient_vectors(
        db, patient_ids, table=table, weighting="binary", dedup="per_patient"
    )
    primary_matrices[table] = (matrix, pids, codes)
    density = matrix.nnz / (matrix.shape[0] * matrix.shape[1])
    print(
        f"{table:20s} matrix={matrix.shape}  nnz={matrix.nnz:,}  "
        f"density={density:.4%}  codes/patient (mean)={matrix.sum(axis=1).mean():.1f}"
    )

## The full 3x3 sweep, condition codes only

All 9 (`weighting`, `dedup`) combinations, run against the same real cohort, to confirm every combination actually produces a usable matrix -- this is the issue's literal done-when check, not just the one primary config.

In [ ]:
sweep_rows = []
for weighting in WEIGHTINGS:
    for dedup in DEDUPS:
        matrix, pids, codes = build_patient_vectors(
            db, patient_ids, table="condition", weighting=weighting, dedup=dedup
        )
        sweep_rows.append(
            {
                "weighting": weighting,
                "dedup": dedup,
                "shape": matrix.shape,
                "nnz": matrix.nnz,
                "mean_value": matrix.data.mean(),
                "max_value": matrix.data.max(),
            }
        )
sweep_df = pd.DataFrame(sweep_rows)
sweep_df

**Note the `binary` rows above**: `nnz` is identical across all three `dedup` values for `binary` weighting, and `mean_value`/`max_value` are both exactly 1.0 in every case. That's not a coincidence -- presence/absence doesn't care how many times or in which visit a code occurred, so `binary` collapses the `dedup` axis to a no-op by construction (proven directly in `tests/test_vectorize.py::test_binary_weighting_is_dedup_invariant`, and re-confirmed here against real data below). The `dedup` choice only actually matters once `weighting` is `count` or `count_normalized`.

In [ ]:
m1, _, c1 = build_patient_vectors(
    db, patient_ids, table="condition", weighting="binary", dedup="per_occurrence"
)
m2, _, c2 = build_patient_vectors(
    db, patient_ids, table="condition", weighting="binary", dedup="per_patient"
)
assert c1 == c2
assert np.allclose(m1.toarray(), m2.toarray())
print("binary weighting is dedup-invariant on real data: confirmed")

## Summary -- issue #4 done-when checklist

- **`build_patient_vectors(weighting, dedup, rollup)` runs against real patient event data** for all 9 `(weighting, dedup)` combinations, on the real `guo_los` cohort (3,850 patients), returning a proper sparse `scipy.sparse.csr_matrix` each time.
- Verified separately per relation type (`condition`, `drug_exposure`, `procedure_occurrence`), matching PROJECT_GUIDE.md's "per relation type" vectorization step.
- Confirmed the `binary`-weighting dedup-invariance property holds on real data, not just the synthetic fixture -- a genuine, testable property of the primary reported configuration, not an assumption.
- `rollup` (the ontology-aware option) is implemented and unit-tested (`tests/test_vectorize.py::test_rollup_merges_codes_before_dedup`) against a synthetic ancestor map; building a *real* `concept_ancestor`-derived map for the full vocabulary is separate follow-up work (PROJECT_GUIDE.md "Can improve" item 1), not required for this issue's done-when.
- Next: issue #5 takes one of these matrices (or a `patient_docs` string built the same way) through TF-IDF + cosine similarity and mutual k-NN sparsification to build the actual patient-patient graph.